# 3 · What a run supplies

A node is a function, so a helper is another function. What only the caller has (who runs the graph, a clock, a client) is a parameter typed `FromRun[T]`, handed in as `compiled.run(from_run={T: value})`.

In [ ]:
from typing import Annotated

from conductor import CompiledGraph, FromRun, GraphRecord, InvalidCallError, NodeRecord, NodeRegistry, Param, Result, node
from conductor.widgets import Textarea

## A node with a helper

In [ ]:
def _words(text: str) -> list[str]:
    return text.split()


@node("word-count", title="Word count", description="Counts the words in a text", category="text")
def word_count(text: Annotated[str, Param(title="Text", widget=Textarea())]) -> Annotated[int, Result(title="Words")]:
    return len(_words(text))


word_count("the quick brown fox")

## A value the run supplies

`Caller` is the host's type. The parameter has no widget, no edge and no value in the record.

In [ ]:
class Caller:
    """Whoever is running the graph: a type the host declares."""

    def __init__(self, name: str) -> None:
        self.name = name


@node("signed", title="Signed", description="Signs a text with the name of whoever ran the graph", category="text")
def signed(
    text: Annotated[str, Param(title="Text", widget=Textarea())],
    who: FromRun[Caller],
) -> Annotated[str, Result(title="Signed")]:
    return f"{text} — {who.name}"


registry = NodeRegistry()
registry.add(word_count, signed)

## A need, not an input

`Interface.needs` lists it. The inputs and the check a call goes through do not.

In [ ]:
interface = signed.versions[1].interface

print("inputs:", [inp.name for inp in interface.inputs])
print("needs: ", sorted(needed.__name__ for needed in interface.needs))
assert "who" not in [inp.name for inp in interface.inputs]

## Run it

In [ ]:
compiled = CompiledGraph.from_graph(
    GraphRecord(
        nodes=[
            NodeRecord(id="note", node="signed", values={"text": "The quick brown fox jumps over the lazy dog"}),
            NodeRecord(id="count", node=word_count.id),
        ],
        edges=[("note.result", "count.text")],
    ),
    registry,
)

results = compiled.results(await compiled.run(from_run={Caller: Caller("Ida")}))
print(results["note"]["result"])
print(results["count"]["result"])

Without it, the call itself raises:

In [ ]:
try:
    await compiled.run()
except InvalidCallError as error:
    print(error)